In [0]:
# Re-run this in Databricks to see the magic happen
display(dbutils.fs.ls("s3://ncf-2026-spring-telescope/dump/"))

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

In [0]:

# Based on the MAGIC Gamma Telescope dataset specs
schema = StructType([
    # StructField(Name, DataType, Nullable)
    StructField("fLength", DoubleType(), True),
    StructField("fWidth", DoubleType(), True),
    StructField("fSize", DoubleType(), True),
    StructField("fConc", DoubleType(), True),
    StructField("fConc1", DoubleType(), True),
    StructField("fAsym", DoubleType(), True),
    StructField("fM3Long", DoubleType(), True),
    StructField("fM3Trans", DoubleType(), True),
    StructField("fAlpha", DoubleType(), True),
    StructField("fDist", DoubleType(), True),
    StructField("class", StringType(), True)
])

In [0]:
# Setup paths
source_path = "s3://ncf-2026-spring-telescope/dump/"
checkpoint_path = "s3://ncf-2026-spring-telescope/_checkpoints/bronze_ingestion/"
bronze_path = "s3://ncf-2026-spring-telescope/bronze/"

# Read the stream using Auto Loader
raw_stream_df = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "false")
    .schema(schema)
    .load(source_path)
)

# Write the stream to Delta Bronze using AvailableNow
query = (
    raw_stream_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(bronze_path)
)

query.awaitTermination()